# 180. KTO：只有 desirable/undesirable 单样本时怎样做偏好优化？

> **面试问题：KTO 与 DPO 的数据合同有何不同？sequence log-ratio、KL reference point、loss aversion 和类别权重怎样实现？**

## 先给结论

DPO 需要同 prompt 的 chosen/rejected pair；KTO 可以使用单个 `(prompt,response,label)` 二元反馈。它把 policy 相对 reference 的 sequence log-ratio 与一个 KL reference point 比较，对 desirable 推高、undesirable 推低，并允许损失厌恶和类别不平衡权重。数据更易收集不代表标签噪声更小。

## 推荐回答主线

1. 先实现 response-only sequence log-prob，明确 prompt/padding mask 与长度归一化选择。
2. 计算 `beta*(logπ-logπ_ref)` 和批量 KL reference point，并阻断 reference/基线梯度。
3. 分别构造 desirable 与 undesirable sigmoid utility，加入类别/loss-aversion 权重。
4. 用梯度方向、标签噪声、长度、KL、独立 win-rate 与安全 slice 验收。

## 教学实现边界

Notebook 实现便于解释的 KTO 核心形式，reference point 用受控 batch 估计；实际论文/库的 KL estimator、权重符号与分布式聚合应以固定版本为准。小 logits 不代表真实偏好质量。

## 一手资料

- [KTO](https://arxiv.org/abs/2402.01306)
- [Direct Preference Optimization](https://arxiv.org/abs/2305.18290)
- [A General Theoretical Paradigm to Understand Learning from Human Preferences](https://arxiv.org/abs/2310.12036)


In [ ]:
import hashlib
import json
import math
from dataclasses import asdict, dataclass

import numpy as np
import warnings
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)

import torch
import torch.nn.functional as F

# 四个独立响应不要求组成同 prompt pair；1=desirable，0=undesirable。
torch.manual_seed(180)
B, T, V = 4, 5, 9
policy_logits = torch.randn(B, T, V, requires_grad=True)
reference_logits = torch.randn(B, T, V)
response_tokens = torch.tensor([[2, 3, 4, 0, 0], [5, 6, 7, 8, 0], [1, 2, 0, 0, 0], [7, 3, 2, 1, 4]])
response_mask = response_tokens.ne(0)
labels = torch.tensor([1, 0, 1, 0], dtype=torch.bool)

assert policy_logits.shape == reference_logits.shape == (B, T, V)
assert response_tokens.shape == response_mask.shape
assert labels.any() and (~labels).any()


## 1. Response sequence log-prob：prompt 与 padding 不进入偏好分数

KTO/DPO 都依赖模型对完整 response 的 log-prob。先对每位置 log-softmax，再 gather 实际 token，只在 response mask 求和。按总和会带长度效应；按均值改变目标，必须显式选择。


In [ ]:
def sequence_log_prob(logits, token_ids, mask, normalize_by_length=False):
    if logits.shape[:-1] != token_ids.shape or token_ids.shape != mask.shape or mask.dtype != torch.bool:
        raise ValueError("logits/token_ids/mask 形状或 dtype 不合法")
    if (~mask.any(-1)).any():
        raise ValueError("每个 response 至少包含一个受训 token")
    token_logp = logits.log_softmax(-1).gather(-1, token_ids[..., None]).squeeze(-1)
    summed = (token_logp * mask).sum(-1)
    if normalize_by_length:
        return summed / mask.sum(-1), token_logp
    return summed, token_logp

# padding logit 不影响序列分数；空 response 直接拒绝而不是悄悄产生零 reward。
policy_logp, token_logp = sequence_log_prob(policy_logits, response_tokens, response_mask)
reference_logp, _ = sequence_log_prob(reference_logits, response_tokens, response_mask)
padding_changed = policy_logits.detach().clone(); padding_changed[~response_mask] += 1000
changed_logp, _ = sequence_log_prob(padding_changed, response_tokens, response_mask)
assert policy_logp.shape == (B,) and token_logp.shape == response_tokens.shape
assert torch.allclose(policy_logp, (token_logp * response_mask).sum(-1))
assert torch.allclose(changed_logp, policy_logp.detach())

empty_response_rejected = False
try:
    bad_mask = response_mask.clone(); bad_mask[0] = False
    sequence_log_prob(policy_logits, response_tokens, bad_mask)
except ValueError:
    empty_response_rejected = True
assert empty_response_rejected


## 2. Implicit reward：相对 reference，而非绝对似然

`r=beta*(logπ-logπ_ref)` 衡量 policy 相对 reference 对该响应的偏移。reference 固定不更新；beta 控制隐式 reward 尺度，不等同学习率。


In [ ]:
def implicit_reward(policy_logp, reference_logp, beta):
    return beta * (policy_logp - reference_logp.detach())

# policy 与 reference 相同则 reward 为零；beta 翻倍 reward 线性翻倍。
beta = 0.1
reward = implicit_reward(policy_logp, reference_logp, beta)
assert reward.shape == labels.shape
assert torch.allclose(implicit_reward(reference_logp, reference_logp, beta), torch.zeros(B))
assert torch.allclose(implicit_reward(policy_logp, reference_logp, 2 * beta), 2 * reward)


## 3. 参考点必须与 sequence reward 使用相同长度尺度

`r(x,y)` 是整段 response 的 log-ratio 求和，所以近似 KL 也先在每条 response 内按 token 求和，再跨样本平均。若一个序列被复制两遍，reward 与 reference point 都应约翻倍；一个求和、一个按 token 平均会制造系统性的长度偏置。


In [ ]:
def kl_reference_point(policy_logits, reference_logits, mask, beta):
    if policy_logits.shape != reference_logits.shape or policy_logits.shape[:-1] != mask.shape:
        raise ValueError("policy/reference/mask 形状不兼容")
    if (~mask.any(-1)).any():
        raise ValueError("KL estimator 不接受空 response")
    policy_log = policy_logits.log_softmax(-1)
    reference_log = reference_logits.log_softmax(-1)
    policy_prob = policy_log.exp()
    token_kl = (policy_prob * (policy_log - reference_log)).sum(-1)
    sequence_kl = (token_kl * mask).sum(-1)
    return (beta * sequence_kl.mean()).detach(), token_kl, sequence_kl

# KL reference point 是 sequence-scale 标量且 stop-gradient；复制 token 轴会与 reward 同比翻倍。
reference_point, token_kl, sequence_kl = kl_reference_point(policy_logits, reference_logits, response_mask, beta)
assert reference_point.requires_grad is False
assert reference_point >= -1e-6
assert torch.isfinite(token_kl).all() and sequence_kl.shape == (B,)

short_policy = policy_logits[:1, :2].detach(); short_reference = reference_logits[:1, :2]
short_tokens = response_tokens[:1, :2]; short_mask = torch.ones_like(short_tokens, dtype=torch.bool)
double_policy = short_policy.repeat(1, 2, 1); double_reference = short_reference.repeat(1, 2, 1)
double_tokens = short_tokens.repeat(1, 2); double_mask = torch.ones_like(double_tokens, dtype=torch.bool)
short_z = kl_reference_point(short_policy, short_reference, short_mask, beta)[0]
double_z = kl_reference_point(double_policy, double_reference, double_mask, beta)[0]
short_reward = implicit_reward(*[sequence_log_prob(value, short_tokens, short_mask)[0] for value in (short_policy, short_reference)], beta)
double_reward = implicit_reward(*[sequence_log_prob(value, double_tokens, double_mask)[0] for value in (double_policy, double_reference)], beta)
assert torch.allclose(double_z, 2 * short_z, atol=1e-6)
assert torch.allclose(double_reward, 2 * short_reward, atol=1e-6)


## 4. KTO utility：好样本追求 reward 超过基线，坏样本追求低于基线

对 desirable 使用 `sigmoid(r-z)`，对 undesirable 使用 `sigmoid(z-r)`；最小化 `1-utility`。这体现相对 gain/loss 的非线性，loss-aversion 可通过类别权重表达。


In [ ]:
def kto_loss(reward, desirable, reference_point, desirable_weight=1.0, undesirable_weight=1.0):
    if reward.shape != desirable.shape or desirable.dtype != torch.bool:
        raise ValueError("reward 与 desirable 标签不兼容")
    if not (math.isfinite(desirable_weight) and math.isfinite(undesirable_weight) and desirable_weight > 0 and undesirable_weight > 0):
        raise ValueError("类别权重必须为有限正数")
    good_utility = torch.sigmoid(reward - reference_point)
    bad_utility = torch.sigmoid(reference_point - reward)
    utility = torch.where(desirable, good_utility, bad_utility)
    weights = torch.where(desirable, torch.full_like(reward, desirable_weight), torch.full_like(reward, undesirable_weight))
    per_example = (1 - utility) * weights
    return per_example.sum() / weights.sum(), utility

# utility 合法；类别权重确实进入逐样本加权和与归一化分母。
kto, utility = kto_loss(reward, labels, reference_point, 1.0, 1.5)
manual_weight = torch.where(labels, torch.ones_like(reward), torch.full_like(reward, 1.5))
assert torch.isfinite(kto) and ((utility > 0) & (utility < 1)).all()
assert torch.allclose(kto, ((1 - utility) * manual_weight).sum() / manual_weight.sum())
better_reward = reward.detach().clone(); better_reward[labels] += 1
assert torch.all(kto_loss(better_reward, labels, reference_point)[1][labels] > kto_loss(reward.detach(), labels, reference_point)[1][labels])

invalid_weight_rejected = False
try:
    kto_loss(reward, labels, reference_point, undesirable_weight=0.0)
except ValueError:
    invalid_weight_rejected = True
assert invalid_weight_rejected


## 5. 梯度方向 oracle：desirable 提 log-ratio，undesirable 降 log-ratio

把 reward 当独立叶子可直接检查损失对它的梯度符号：好样本梯度为负，梯度下降会增 reward；坏样本梯度为正，会降 reward。这是实现公式时极有价值的单测。


In [ ]:
def kto_from_logits(policy_logits, reference_logits, token_ids, mask, desirable, beta, desirable_weight, undesirable_weight):
    policy_sequence, _ = sequence_log_prob(policy_logits, token_ids, mask)
    reference_sequence, _ = sequence_log_prob(reference_logits, token_ids, mask)
    sequence_reward = implicit_reward(policy_sequence, reference_sequence, beta)
    z0, _, sequence_kl = kl_reference_point(policy_logits, reference_logits, mask, beta)
    loss, utility = kto_loss(sequence_reward, desirable, z0, desirable_weight, undesirable_weight)
    return loss, {"reward": sequence_reward, "reference_point": z0, "sequence_kl": sequence_kl, "utility": utility}

# 端到端主路径从 logits 聚合 sequence reward/KL，再把真实类别权重送入 KTO loss。
policy_probe = policy_logits.detach().clone().requires_grad_(True)
reference_probe = reference_logits.detach().clone().requires_grad_(True)
end_to_end_loss, diagnostics = kto_from_logits(
    policy_probe, reference_probe, response_tokens, response_mask, labels, beta, 1.0, 1.5
)
manual_loss, _ = kto_loss(diagnostics["reward"], labels, diagnostics["reference_point"], 1.0, 1.5)
unweighted_loss = kto_from_logits(
    policy_probe, reference_probe, response_tokens, response_mask, labels, beta, 1.0, 1.0
)[0]
assert torch.allclose(end_to_end_loss, manual_loss)
assert not torch.allclose(end_to_end_loss, unweighted_loss)
end_to_end_loss.backward()
assert policy_probe.grad is not None and torch.isfinite(policy_probe.grad).all() and policy_probe.grad.norm() > 0
assert reference_probe.grad is None  # reference reward 与 z0 均 stop-gradient。

# 独立 reward 叶子验证好样本梯度向上、坏样本梯度向下。
reward_leaf = reward.detach().clone().requires_grad_(True)
direction_loss, _ = kto_loss(reward_leaf, labels, reference_point)
direction_loss.backward()
assert torch.all(reward_leaf.grad[labels] < 0)
assert torch.all(reward_leaf.grad[~labels] > 0)
assert torch.isfinite(reward_leaf.grad).all()


## 6. 类别不平衡：权重按目标分布设计，不靠重复采样掩盖

点赞/点踩数据常极不平衡，还带展示偏差。权重可让两类总贡献平衡，但不能修复缺失 support 或标签噪声；日志保留原始 prevalence、propensity 和采样策略。


In [ ]:
def balanced_class_weights(binary_labels):
    positive = int(binary_labels.sum())
    negative = len(binary_labels) - positive
    if positive == 0 or negative == 0:
        raise ValueError("估计平衡权重需要两个类别都出现")
    return len(binary_labels) / (2 * positive), len(binary_labels) / (2 * negative)

# 平衡数据权重均为 1；稀有正类权重更大且送入真实 kto_loss 后等于显式加权公式。
good_w, bad_w = balanced_class_weights(labels)
rare_labels = torch.tensor([1, 0, 0, 0, 0], dtype=torch.bool)
rare_reward = torch.tensor([0.8, -0.4, 0.2, -1.0, 0.5])
rare_good_w, rare_bad_w = balanced_class_weights(rare_labels)
rare_loss, rare_utility = kto_loss(rare_reward, rare_labels, torch.tensor(0.1), rare_good_w, rare_bad_w)
rare_weights = torch.where(rare_labels, torch.full_like(rare_reward, rare_good_w), torch.full_like(rare_reward, rare_bad_w))
assert good_w == bad_w == 1.0
assert rare_good_w > rare_bad_w and math.isclose(rare_good_w, 4 * rare_bad_w)
assert torch.allclose(rare_loss, ((1 - rare_utility) * rare_weights).sum() / rare_weights.sum())

single_class_rejected = False
try:
    balanced_class_weights(torch.ones(3, dtype=torch.bool))
except ValueError:
    single_class_rejected = True
assert single_class_rejected


## 7. 噪声与长度切片：unpaired 更便宜，但上下文更难对齐

binary label 可能只评价局部属性、受用户心情或 response 长度影响。按 prompt/user/time 分组切分，检查 desirable rate、长度、语言、安全类别与来源；同用户泄漏会虚高。


In [ ]:
def grouped_split(group_ids, holdout_groups):
    return np.array([group not in holdout_groups for group in group_ids], dtype=bool)

def label_length_report(labels, lengths):
    labels = np.asarray(labels, dtype=bool); lengths = np.asarray(lengths)
    return float(lengths[labels].mean()), float(lengths[~labels].mean())

# 同 group 整体进入 holdout；报告能暴露好坏标签长度差。
groups = ["user-a", "user-a", "user-b", "user-c"]
train_mask = grouped_split(groups, {"user-a"})
good_len, bad_len = label_length_report(labels.numpy(), response_mask.sum(-1).numpy())
assert train_mask.tolist() == [False, False, True, True]
assert good_len > 0 and bad_len > 0
assert train_mask.sum() == 2


## 8. 发布门禁：KTO recipe、reference 与数据曝光策略共同版本化

权重只是一部分；还要绑定 reference checkpoint、chat template、response mask、beta、KL estimator、类别权重和数据采样。验收比较独立人评/执行指标、KL、长度、拒答、安全 slice 与成本。


In [ ]:
@dataclass(frozen=True)
class KTOArtifact:
    policy_base: str
    reference: str
    beta: float
    kl_estimator: str
    response_mask: str
    class_weighting: str

def artifact_hash(artifact):
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()

# artifact 明确记录 sequence-sum KL、空响应拒绝与真实类别加权合同。
artifact = KTOArtifact("base-v4", "reference-v4", beta, "sequence-forward-kl-sum-v2", "assistant-nonempty-v3", "positive-balanced-v2")
digest = artifact_hash(artifact)
assert artifact.policy_base != "" and artifact.kl_estimator.startswith("sequence-")
assert len(digest) == 64
assert digest != artifact_hash(KTOArtifact("base-v4", "reference-v4", 0.2, artifact.kl_estimator, artifact.response_mask, artifact.class_weighting))


## 面试收束：从公式走到生产合同

建议用六步回答：目标与约束、张量/数据合同、核心公式、正确性反例、质量—成本评测、版本与回滚。Notebook 的小模型只证明机制和边界，不代表论文规模结果、真实 GPU kernel 加速或线上泛化。生产替换时仍应保留同一批 oracle，并补齐目标硬件 profiling、分布式一致性、数据 provenance、安全审计和灰度发布。

继续追问时要主动区分：训练期方法与已有 checkpoint 的后处理、理论 FLOPs 与 wall-clock、平均质量与关键 slice、可逆近似与不可逆状态、模型置信与校准后的决策概率。
